In [2]:
import pandas as pd
import math
import random
from pathlib import Path
from collections import defaultdict

# 1. Flexible Paths Configuration
BASE_DIR = Path.cwd().parent
if not (BASE_DIR / "data" / "processed").exists():
    BASE_DIR = Path.cwd()

ELO_PATH = BASE_DIR / "data" / "processed" / "current_elo_ratings.csv"
OUTPUT_REPORT_PATH = BASE_DIR / "data" / "processed" / "monte_carlo_results.csv"

df_elo = pd.read_csv(ELO_PATH)
elo_dict = dict(zip(df_elo['team'], df_elo['current_elo']))

# 2. Dynamic Probability Engine
def get_win_probability(team_a, team_b):
    elo_a = elo_dict.get(team_a, 1500)
    elo_b = elo_dict.get(team_b, 1500)
    return 1.0 / (1.0 + math.pow(10, (elo_b - elo_a) / 400.0))

def simulate_match(team_a, team_b):
    prob_a = get_win_probability(team_a, team_b)
    return team_a if random.random() < prob_a else team_b

# 3. FULL Topology (Round of 32)
r32_matches = {
    # LEFT
    'L1': ('Germany', 'Paraguay'), 'L2': ('France', 'Sweden'),
    'L3': ('South Africa', 'Canada'), 'L4': ('Netherlands', 'Morocco'),
    'L5': ('Portugal', 'Croatia'), 'L6': ('Spain', 'Austria'),
    'L7': ('United States', 'Bosnia and Herzegovina'), 'L8': ('Belgium', 'Senegal'),
    # RIGHT
    'R1': ('Brazil', 'Japan'), 'R2': ('Ivory Coast', 'Norway'),
    'R3': ('Mexico', 'Ecuador'), 'R4': ('England', 'DR Congo'),
    'R5': ('Argentina', 'Cape Verde'), 'R6': ('Australia', 'Egypt'),
    'R7': ('Switzerland', 'Algeria'), 'R8': ('Colombia', 'Ghana')
}

paths = {
    # Round of 16 Paths
    'R16_L1': ('L1', 'L2'), 'R16_L2': ('L3', 'L4'),
    'R16_L3': ('L5', 'L6'), 'R16_L4': ('L7', 'L8'),
    'R16_R1': ('R1', 'R2'), 'R16_R2': ('R3', 'R4'),
    'R16_R3': ('R5', 'R6'), 'R16_R4': ('R7', 'R8'),
    # Quarter-Final Paths
    'QF_L1': ('R16_L1', 'R16_L2'), 'QF_L2': ('R16_L3', 'R16_L4'),
    'QF_R1': ('R16_R1', 'R16_R2'), 'QF_R2': ('R16_R3', 'R16_R4'),
    # Semi-Final Paths
    'SF_L': ('QF_L1', 'QF_L2'), 'SF_R': ('QF_R1', 'QF_R2'),
    # Final Path
    'FINAL': ('SF_L', 'SF_R')
}

# 4. Monte Carlo Engine
SIMULATIONS = 10000
champions = defaultdict(int)
finalists = defaultdict(int)
semifinalists = defaultdict(int)

print(f"[INFO] Running {SIMULATIONS} parallel realities on the FULL 32-team bracket...")

for _ in range(SIMULATIONS):
    winners = {}
    
    # R32
    for match_id, (home, away) in r32_matches.items():
        winners[match_id] = simulate_match(home, away)
        
    # Subsequent Rounds
    for match_id, (m_a, m_b) in paths.items():
        winners[match_id] = simulate_match(winners[m_a], winners[m_b])
        if match_id.startswith('QF'):
            semifinalists[winners[match_id]] += 1
        elif match_id.startswith('SF'):
            finalists[winners[match_id]] += 1
            
    # The Grand Final
    champion = simulate_match(winners[paths['FINAL'][0]], winners[paths['FINAL'][1]])
    champions[champion] += 1

# 5. Output
results = []
for team in set(list(champions.keys()) + list(finalists.keys()) + list(semifinalists.keys())):
    results.append({
        'Team': team,
        'Win_Probability': (champions[team] / SIMULATIONS) * 100,
        'Reach_Final_Prob': (finalists[team] / SIMULATIONS) * 100,
        'Reach_Semi_Prob': (semifinalists[team] / SIMULATIONS) * 100
    })

df_results = pd.DataFrame(results).sort_values(by='Win_Probability', ascending=False)
df_results.to_csv(OUTPUT_REPORT_PATH, index=False)

print("\n" + "="*60)
print(" MONTE CARLO SIMULATION COMPLETE (FULL DYNAMIC ELO)")
print("="*60)
df_display = df_results.head(10).copy()
for col in ['Win_Probability', 'Reach_Final_Prob', 'Reach_Semi_Prob']:
    df_display[col] = df_display[col].round(1).astype(str) + '%'
print(df_display.to_string(index=False))

[INFO] Running 10000 parallel realities on the FULL 32-team bracket...

 MONTE CARLO SIMULATION COMPLETE (FULL DYNAMIC ELO)
       Team Win_Probability Reach_Final_Prob Reach_Semi_Prob
  Argentina           24.2%            39.1%           57.0%
     France           16.0%            26.9%           41.9%
      Spain           13.8%            23.8%           39.5%
     Brazil            6.0%            11.6%           24.5%
   Colombia            5.5%            12.2%           23.0%
    England            4.5%            10.3%           23.8%
     Mexico            4.3%             9.4%           19.6%
    Morocco            3.7%             8.0%           16.6%
   Portugal            3.7%             8.2%           17.3%
Netherlands            3.6%             8.2%           16.1%
